In [ ]:
import pandas as pd
import holoviews as hv
import panel as pn
import geoviews as gv
from holoviews import opts
from bokeh.models import HoverTool

hv.extension('bokeh')
pn.extension()

In [ ]:
from dotenv import load_dotenv
import os 

load_dotenv("/home/jegesm/Dropbox/data-ex-vis/Lectures/Interactive_Visualization/env_variables")

In [ ]:
DATADIR = os.getenv("DATADIR")
df = pd.read_csv(f'{DATADIR}/traffic-violation-small.csv', parse_dates=['Date Of Stop','Time Of Stop'])

In [ ]:
# Add 'All' option to dropdowns
make_dropdown = pn.widgets.Select(name='Make', options=['All'] + df['Make'].unique().tolist(), value='All')
color_dropdown = pn.widgets.Select(name='Color', options=['All'] + df['Color'].unique().tolist(), value='All')
gender_dropdown = pn.widgets.Select(name='Gender', options=['All'] + df['Gender'].unique().tolist(), value='All')

# Create the map of locations using Latitude and Longitude
def get_map(make, color, gender):
    filtered_df = df.copy()
    
    # Apply filtering based on dropdown selections
    if make != 'All':
        filtered_df = filtered_df[filtered_df['Make'] == make]
    if color != 'All':
        filtered_df = filtered_df[filtered_df['Color'] == color]
    if gender != 'All':
        filtered_df = filtered_df[filtered_df['Gender'] == gender]
    
    # Ensure that there is data after filtering
    if filtered_df.empty:
        return gv.Text(x=0, y=0, text='No data for this selection', fontsize=15)
    
    # Create a GeoDataFrame for the locations
    points = gv.Points(filtered_df, kdims=['Longitude', 'Latitude'], vdims=['Location', 'Description'])
    points.opts(
        size=5, color='red', tools=['hover'], width=800, height=600, title="Map of Stops"
    )
    
    return points

# Create a bar chart for the accidents and other attributes
def get_bar_chart(make, color, gender):
    filtered_df = df.copy()
    
    # Apply filtering based on dropdown selections
    if make != 'All':
        filtered_df = filtered_df[filtered_df['Make'] == make]
    if color != 'All':
        filtered_df = filtered_df[filtered_df['Color'] == color]
    if gender != 'All':
        filtered_df = filtered_df[filtered_df['Gender'] == gender]
    
    # Ensure that there is data after filtering
    if filtered_df.empty:
        return hv.Bars([]).opts(
            xlabel='Accident Type', ylabel='Count', title='Accident Data', height=400, width=600
        )
    
    # Aggregate data for the accident types
    accident_data = filtered_df[['Accident', 'Belts', 'Personal Injury', 'Property Damage', 'Fatal']].sum()
    
    bar_chart = hv.Bars(accident_data).opts(
        xlabel='Accident Type', ylabel='Count', title='Accident Data', height=400, width=600
    )
    
    return bar_chart

# Set up a layout and connect the dropdowns to update the map and chart
@pn.depends(make=make_dropdown.param.value, color=color_dropdown.param.value,
            gender=gender_dropdown.param.value)
def update_dashboard(make, color, gender):
    map_view = get_map(make, color, gender)
    bar_chart = get_bar_chart(make, color, gender)
    
    return pn.Column(
        pn.Row(make_dropdown, color_dropdown, gender_dropdown),
        pn.Row(map_view),
        pn.Row(bar_chart)
    )

In [ ]:
# Create the dashboardAttributeError: 'function' object has no attribute 'servable'
dashboard = update_dashboard(make_dropdown, color_dropdown, gender_dropdown)

In [ ]:
dashboard.servable()